# Western Balkan Studies - LANDs plots

## Define RUN TAG

In [ ]:
RUN_ID = 'vre_low_20260310'

* load packages

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import RES.visuals as vis
from RES import utility as utils
from RES.hdf5_handler import DataHandler
import RES.utility as utlis
import RES.lands as lands

plt.style.use('../RES/visual_styles/elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
cfg=utils.load_config('../config/config_WB6.yaml')
run_id=cfg.get('Scenario').get('run_id')


sub_national_unit_tag=cfg.get('GADM').get('datafield_mapping').get('NAME_2')
country_name=cfg.get('country','Western Balkan Region') # type: ignore
country_kwd=country_name.replace(' ','')
CRS_m = cfg.get('default_CRS').get('meters')  # Default metric CRS
CRS_d = cfg.get('default_CRS').get('degrees')  # Default geographic CRS
regions=['AL','BA','XK','ME','RS','MK']  #'AL','BA','XK','ME','MK','RS'

vis_save_to_root=utils.ensure_path(f"../vis/{country_kwd}/{RUN_ID}/WB6_fullRegion")

## Load Store

In [ ]:
 #All the regions should have RUN_ID results available
WB6_store = {}
utils.print_update(level=1,message=f"Loading data stores for WB6 regions with RUN_ID: {RUN_ID} and Regions: {regions}")
for region in regions:
    country_dict = {}
    try:
        store = Path(f"../data/store/{country_kwd}/resources_{country_kwd}_{region}_{RUN_ID}.h5")
        assert store.exists(), f"Store path doesn't exist: {store}"
        res_data = DataHandler(store, show_structure=False)
        country_dict['cells'] = res_data.from_store('cells')
        country_dict['boundary'] = res_data.from_store('boundary')
        country_dict['lines'] = res_data.from_store('lines')
        country_dict['LandAvailability'] = res_data.from_store('LandAvailability')
        WB6_store[region] = country_dict
        utils.print_update(level=2,message=f"✓ Loaded data for region: {region}") 
    except Exception as e:
        print(f"X Error with region {region}: {e}")
        continue

## Load All Cells

In [ ]:
all_cells_df=[WB6_store[region]['cells'] for region in WB6_store]
WB6_cells = gpd.GeoDataFrame(pd.concat(all_cells_df, ignore_index=True), crs=all_cells_df[0].crs)

# Load Test/Validation data

In [ ]:
existing_VREs_data_path=Path("../data/validation_data/existing_VREs_WB6.csv")
if existing_VREs_data_path.exists():
    existing_VREs=pd.read_csv(existing_VREs_data_path)
    existing_VREs_gdf=gpd.GeoDataFrame(existing_VREs,geometry=gpd.points_from_xy(existing_VREs.Longitude,existing_VREs.Latitude),crs="EPSG:4326")
    utils.print_update(level=1,message=f"✓ Loaded validation data for existing VREs from {existing_VREs_data_path}")
else:
    existing_VREs_gdf=None
    utils.print_warning(f"Validation data for existing VREs not found at {existing_VREs_data_path}")

# Lands

- Prepare WB6 boundary

In [ ]:
# Combine all region boundaries into a single GeoDataFrame
boundary_gdfs = [WB6_store[region]['boundary'] for region in WB6_store]
WB6_boundary = gpd.GeoDataFrame(pd.concat(boundary_gdfs, ignore_index=True), crs=boundary_gdfs[0].crs)
WB6_boundary_dissolved = WB6_boundary.dissolve(by="Country")[["geometry"]].reset_index()

- Process the boundary info for raster plotting

In [ ]:
WB6_boundary_dissolved_reproj=WB6_boundary_dissolved.to_crs(CRS_m)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = WB6_boundary_dissolved_reproj.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

* Country Map (base)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

fig, ax = plt.subplots(figsize=(12, 7))
fig.suptitle(f" {country_name}", fontsize=16, fontweight='bold')

WB6_boundary_dissolved_reproj.plot(
    column="Country",
    categorical=True,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.5,
    ax=ax
)

for _, row in WB6_boundary_dissolved_reproj.iterrows():
    point = row.geometry.representative_point()
    txt = ax.annotate(
        row["Country"],
        xy=(point.x, point.y),
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="black"
    )
    txt.set_path_effects([
        pe.withStroke(linewidth=2, foreground="white")
    ])

ax.set_axis_off()
plt.tight_layout()
plt.savefig(f"{vis_save_to_root}/{country_name}_map.png")

- Load All availabilities

In [ ]:
all_land_availabilities_gdfs=[WB6_store[region]['LandAvailability'] for region in WB6_store]
WB6_LandAvailability = gpd.GeoDataFrame(pd.concat(all_land_availabilities_gdfs, ignore_index=True), crs=all_land_availabilities_gdfs[0].crs)

if WB6_LandAvailability.crs != CRS_m:
    WB6_LandAvailability_plot = WB6_LandAvailability.to_crs(CRS_m)
else:
    WB6_LandAvailability_plot = WB6_LandAvailability

- Merge Availability info to WB6 store cells

In [ ]:
WB6_LandAvailability = WB6_LandAvailability.to_crs(WB6_cells.crs)

land_cent = WB6_LandAvailability[["LandAvailability_wind", "LandAvailability_solar", "geometry"]].copy()
land_cent["geometry"] = land_cent.geometry.centroid

WB6_cells = gpd.sjoin(
    WB6_cells,
    land_cent,
    how="left",
    predicate="contains"
).drop(columns="index_right")

- Create cells' instance for plotting (CRS-m)

In [ ]:
if WB6_cells.crs != CRS_m:
    WB6_cells_plot = WB6_cells.to_crs(CRS_m)
else:
    WB6_cells_plot = WB6_cells

- Plot combined Availability

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib as mpl

# Convert to percent
WB6_LandAvailability_plot["LandAvailability_solar_pct"] = WB6_LandAvailability_plot["LandAvailability_solar"] * 100
WB6_LandAvailability_plot["LandAvailability_wind_pct"] = WB6_LandAvailability_plot["LandAvailability_wind"] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=300)
fig.suptitle(f"{country_name}: Land Availability for VRE development",
             fontsize=16, fontweight="bold", y=0.99)

plot_specs = [
    ("LandAvailability_solar_pct", "Solar", axes[0]),
    ("LandAvailability_wind_pct", "Wind", axes[1]),
]

cmap = plt.cm.get_cmap("YlGn",6)
norm = mpl.colors.Normalize(vmin=0, vmax=100)

for col, panel_title, ax in plot_specs:
    WB6_LandAvailability_plot.plot(
        column=col,
        cmap=cmap,
        edgecolor="white",
        linewidth=0.3,
        legend=False,
        vmin=0,
        vmax=100,
        ax=ax,
    )

    WB6_boundary_dissolved_reproj.plot(
        color="none",
        edgecolor="k",
        linewidth=0.5,
        ax=ax
    )

    for _, row in WB6_boundary_dissolved_reproj.iterrows():
        point = row.geometry.representative_point()
        txt = ax.annotate(
            row["Country"],
            xy=(point.x, point.y),
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold",
            color="black"
        )
        txt.set_path_effects([
            pe.withStroke(linewidth=2.5, foreground="white")
        ])

    ax.set_title(panel_title, fontsize=14, fontweight="bold")
    ax.set_axis_off()

# Manually adjust map area to leave room at bottom
fig.subplots_adjust(bottom=0.1,wspace=0.08)

# Dedicated colorbar axis: [left, bottom, width, height]
cax = fig.add_axes([0.2, 0.02, 0.6, 0.025])

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
cbar.set_label("Land availability (%)", fontsize=14, fontweight="bold")
cbar.ax.tick_params(labelsize=12)
cbar.set_ticks([0, 20, 40, 60, 80, 100])

plt.savefig(
    f"{vis_save_to_root}/{country_name}_map_LandAvailability_solar_wind.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Corine Land Cover

- Load CLC raster config to extract file name and other attributes

In [ ]:
for raster in cfg.get('CORINE').get('raster_types'):
    if raster['name']=='CORINE_land_cover':
        CLC_raster_cfg=raster

- Create a WB6 clipped (to geom) raster for plotting purpose. Our main workflow gives Country specific rasters.

 > Clip Raster to WB6 boundary (otherwise raster distribution will show wrong numbers)

In [ ]:
CLC_source_raster_path=Path(f"../{CLC_raster_cfg['root']}/{CLC_raster_cfg['raster']}")

 - Clip CLC to Wb6 | Method 1
   - Load complete CLC data, use rio clip with reprojected (CRS_m) boundary  

!!! METHOD 1: Provides error and missing classes due to clipping logic

In [ ]:
CLC_da=utils.get_raster_da(CLC_source_raster_path) 

In [ ]:
# Ensure same CRS
# if WB6_boundary_dissolved_reproj.crs != CLC_da.rio.crs:
#     WB6_boundary_dissolved_reproj = WB6_boundary_dissolved_reproj.to_crs(CLC_da.rio.crs)

# Clip raster to boundary
# CLC_raster_WB6_da = CLC_da.rio.clip(WB6_boundary_dissolved_reproj.geometry,
#                                     # WB6_boundary_dissolved_reproj.crs,
#                                     drop=False,
#                                     invert=False,
#                                     all_touched=True)

 - Clip CLC to Wb6 | Method 2 (if more alteration necessary)
   - More Control to alter the resolution and CRS 

In [ ]:
CLC_WB6_raster_path = lands.clip_to_boundary_and_resample_raster(in_raster_config= CLC_raster_cfg,
                                           boundary_name='WB6',
                                           boundary=WB6_boundary_dissolved_reproj,
                                           CRS_meters=CRS_m,
                                           source_raster_path=CLC_source_raster_path,
                                           )

In [ ]:
CLC_raster_WB6_da=utils.get_raster_da(CLC_WB6_raster_path)

In [ ]:
utils.check_raster_classes(CLC_da,CLC_raster_WB6_da,WB6_boundary_dissolved_reproj)

- Load Legends and Raster layers attributes from config

In [ ]:
CLC_legends=pd.read_csv("../data/legends/CLC_2018_legend.csv")
class_inclusion_layers:dict=cfg.get('CORINE').get('raster_types')[0]['class_inclusion']

- Check summary of existing sites

In [ ]:
existing_VREs_gdf_with_landcover,existing_VREs_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=CLC_raster_WB6_da,
    legend_df=CLC_legends,
    class_col_name="CLC_landcover",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=existing_VREs_summary,
    class_col="CLC_landcover_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Land-Cover Class and Technology",
    figsize=(7, 3.5),
    wrap_width=25,
    fontsize=6,
    colors=["#d6851c", "#6f32e0"],
    save_to=vis_save_to_root/f"existing_VREs_by_landcover_{country_kwd}.png"
)


In [ ]:
LandCover_CLC_class_distribution=lands.plot_raster_class_distribution(CLC_raster_WB6_da,
                                     CLC_legends,
                                     show=True,
                                     figsize=(8,5),
                                     save_path=vis_save_to_root/f'CLC_class_distribution_{country_kwd}.png',
                                     pct_threshold=0.2)

- Define layers for plotting

In [ ]:
layers_included:dict=CLC_raster_cfg['class_inclusion']

- Plot CLC + Existing VREs + Boundaries

In [ ]:
classes_to_plot=layers_included # None , plots all layers

if classes_to_plot is None:
    title = "CLC 2018 - All Layers"
    save_to_path = vis_save_to_root/"CLC_AllLayers.png"
else:
    title = f"CLC 2018 - {country_kwd} - suitable for New VRE Sites"
    save_to_path = vis_save_to_root/f"CLC_{country_kwd}_forNewVREsites.png"

fig1,ax1,save_to1=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=CLC_da,
    raster_legends=CLC_legends,
    classes_to_plot=classes_to_plot, #layers,                 # <- all classes
    boundary=WB6_boundary_dissolved_reproj,
    existing_VREs_gdf=existing_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=10.0,
    figsize=(16, 9),
    dpi=500,
    marker_highlight_width=3,
    title=title,
    output_path=save_to_path,
    legend_anchor=(.98, .99),
    legend_fontsize=10
)

utils.print_update(level=1,message=f"plot saved to: {save_to_path}")

## GAEZ

### Terrain

In [ ]:
GAEZ_terrain_raster_path='../data/downloaded_data/GAEZ/Rasters_in_use/LR/ter/slpmed30s.tif'
GAEZ_terrain_raster_legends=pd.read_csv("../data/legends/gaez_terrains_legend.csv")
GAEZ_terrain_cfg= next((item for item in cfg.get('GAEZ').get('raster_types') if 'terrain_resources' in item.get('name', '')), None)

- Load Raster Data as data-array

In [ ]:
GAEZ_terrain_raster_da = utils.get_raster_da(GAEZ_terrain_raster_path) 

- Clip Raster to WB6 boundary (otherwise raster distribution will show wrong numbers)

In [ ]:
GAEZ_terrain_raster_da_clipped=utils.get_raster_da(lands.clip_to_boundary_and_resample_raster(in_raster_config=           GAEZ_terrain_cfg,
                                           boundary_name='WB6',
                                           boundary=WB6_boundary_dissolved_reproj,
                                           CRS_meters=CRS_m,
                                           source_raster_path=GAEZ_terrain_raster_path,
                                           ))

In [ ]:
# Ensure same CRS
if WB6_boundary_dissolved.crs != GAEZ_terrain_raster_da.rio.crs:
    WB6_boundary_GAEZ = WB6_boundary_dissolved.to_crs(GAEZ_terrain_raster_da.rio.crs)
else:
    WB6_boundary_GAEZ = WB6_boundary_dissolved
# Clip raster to boundary
GAEZ_terrain_raster_da_clipped = GAEZ_terrain_raster_da.rio.clip(WB6_boundary_GAEZ.geometry, WB6_boundary_GAEZ.crs)

- Review class distributions

In [ ]:
lands.plot_raster_class_distribution(GAEZ_terrain_raster_da_clipped,
                                     GAEZ_terrain_raster_legends,
                                     show=True,
                                     figsize=(8,3),
                                     save_path=vis_save_to_root/f'GAEZ_terrains_class_distribution_{country_kwd}.png')


- Review layers mapping to existing VREs

In [ ]:
existing_VREs_gdf_with_terrains,terrains_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=GAEZ_terrain_raster_da_clipped,
    legend_df=GAEZ_terrain_raster_legends,
    class_col_name="GAEZ_terrain",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=terrains_summary,
    class_col="GAEZ_terrain_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Terrain Class and Technology",
    figsize=(6, 2),
    wrap_width=25,
    fontsize=6,
    colors=["#d6851c", "#6f32e0"],
    save_to=vis_save_to_root/f"existing_VREs_by_terrain_{country_kwd}.png"
)


- Define the layers to be plotted

In [ ]:
layers_excluded:dict=GAEZ_terrain_cfg['class_exclusion']

# Unique CLC codes in your clipped area
unique_classes = np.unique(GAEZ_terrain_raster_da_clipped.values[~np.isnan(GAEZ_terrain_raster_da_clipped.values)])

layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}

- Plot GAEZ Terrain with defined layers and existing VREs

In [ ]:
classes_to_plot = layers_included # layers_included #None , plots all layers

if classes_to_plot is None:
    GAEZ_terrain_save_to_path=vis_save_to_root/f"GAEZ_terrains_{country_kwd}_allLayers.png"
    title = "GAEZ Terrains - All Layers"
else:
# Update DOC contents (if needed)
    GAEZ_terrain_save_to_path=vis_save_to_root/f"GAEZ_terrains_{country_kwd}_forNewVREsites.png"
    title = "GAEZ Terrains - For New VRE Sites"

fig2,ax2,save_to2=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=GAEZ_terrain_raster_da_clipped,
    raster_legends=GAEZ_terrain_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=WB6_boundary_dissolved,
    existing_VREs_gdf=existing_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=5.0,
    legend_fontsize=11,
    title=title,
    output_path=GAEZ_terrain_save_to_path,
    legend_anchor=(.1, .4),
    marker_highlight_width=3,
)
utils.print_update(level=1,message=f"plot saved to: {GAEZ_terrain_save_to_path}")

### Exclusion

In [ ]:
GAEZ_exclusion_raster_path = "../data/downloaded_data/GAEZ/Rasters_in_use/LR/excl/exclusion_2017.tif"
GAEZ_exclusion_raster_legends=pd.read_csv("../data/legends/gaez_exclusion_legend.csv")
GAEZ_exclusion_cover_cfg= next((item for item in cfg.get('GAEZ').get('raster_types') if 'exclusion_areas' in item.get('name', '')), None)

- Load Raster Data as data-array

In [ ]:
GAEZ_exclusion_raster_data = utils.get_raster_da(GAEZ_exclusion_raster_path)

- Clip to WB6 Boundary

In [ ]:
# Ensure same CRS
if WB6_boundary_dissolved.crs != GAEZ_exclusion_raster_data.rio.crs:
    WB6_boundary_dissolved = WB6_boundary_dissolved.to_crs(GAEZ_exclusion_raster_data.rio.crs)

# Clip raster to boundary
GAEZ_exclusion_raster_data_clipped = GAEZ_exclusion_raster_data.rio.clip(WB6_boundary_dissolved.geometry, WB6_boundary_dissolved.crs)

- Plot Class Distribution

In [ ]:
lands.plot_raster_class_distribution(GAEZ_exclusion_raster_data_clipped,
                                     GAEZ_exclusion_raster_legends,
                                     show=True,
                                     figsize=(8,3),
                                     save_path=vis_save_to_root/f'GAEZ_Exclusion_class_distribution_{country_kwd}.png')

- Review existing sites and excluded lands

In [ ]:
existing_VREs_gdf_with_exclusions,exclusions_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=GAEZ_exclusion_raster_data_clipped,
    legend_df=GAEZ_exclusion_raster_legends,
    class_col_name="GAEZ_exclusion",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=exclusions_summary,
    class_col="GAEZ_exclusion_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Exclusion Class and Technology",
    figsize=(8, 2),
    wrap_width=25,
    fontsize=8,
    colors=["#d6851c", "#6f32e0"],
    save_to=vis_save_to_root/f"existing_VREs_by_exclusions_{country_kwd}.png"
)

- Define layers to be plotted

In [ ]:
layers_excluded:dict=GAEZ_exclusion_cover_cfg['class_exclusion']

# Unique GAEZ terrain codes in your clipped area
unique_classes = np.unique(GAEZ_exclusion_raster_data.values[~np.isnan(GAEZ_exclusion_raster_data.values)])

layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}

- Plot

In [ ]:
classes_to_plot = layers_included  #None , plots all layers

if classes_to_plot is None:
    GAEZ_exclusion_save_to_path=vis_save_to_root/f"GAEZ_exclusion_{country_kwd}_allLayers.png"
    title = "GAEZ Exclusions - All Layers"
else:
# Update DOC contents (if needed)
    GAEZ_exclusion_save_to_path=vis_save_to_root/f"GAEZ_exclusion_{country_kwd}_forNewVREsites.png"
    title = "GAEZ Exclusions - For New VRE Sites"

fig3,ax3,save_to3=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=GAEZ_exclusion_raster_data_clipped,
    raster_legends=GAEZ_exclusion_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=WB6_boundary_dissolved,
    existing_VREs_gdf=existing_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=5.0,
    figsize=(16, 9),
    dpi=500,
    legend_fontsize=12,
    title=title,
    output_path=GAEZ_exclusion_save_to_path,
    legend_anchor=(.1, .3),
    marker_highlight_width=3,
)
utils.print_update(level=1,message=f"plot saved to: {GAEZ_exclusion_save_to_path}")

# Grid

In [ ]:
all_lines=[WB6_store[region]['lines'] for region in WB6_store]
WB6_lines = gpd.GeoDataFrame(pd.concat(all_lines, ignore_index=True), crs=all_lines[0].crs)

In [ ]:
# Convert to numeric
WB6_lines['voltage_kv'] = pd.to_numeric(WB6_lines['voltage'], errors='coerce') / 1000

# Define voltage bins
bins = [0, 12, 25, 132, 220, float("inf")]
labels = ["<12 kV", "12–25 kV", "25–132 kV", "132–220 kV", "≥220 kV"]
WB6_lines['voltage_class'] = pd.cut(WB6_lines['voltage_kv'], bins=bins, labels=labels, right=False)


In [ ]:
WB6_lines

- Plot gird

In [ ]:
ax=WB6_boundary_dissolved.plot(edgecolor='black', facecolor='grey', linewidth=0.2, figsize=(10, 8),alpha=0.1)

ax.set_axis_off()

if existing_VREs_gdf is not None and not existing_VREs_gdf.empty:

    existing_VREs_gdf[existing_VREs_gdf['Technology'].str.lower()=='wind'].plot(ax=ax, color='None', edgecolor='blue',markersize=60, linewidth=0.8,marker='o', label='Existing VREs',alpha=1,zorder=2)

    # Add legend for existing wind (purple)
    legend_handles = [Patch(facecolor='none', edgecolor='purple', label='Existing Wind', alpha=1)]
    ax.legend(handles=legend_handles, loc='upper left', fontsize=8, frameon=False)

    existing_VREs_gdf[existing_VREs_gdf['Technology'].str.lower()=='solar'].plot(ax=ax, color='None', edgecolor='orangered',markersize=60,linewidth=0.8, marker='o', label='Existing VREs',alpha=1,zorder=2)
WB6_lines.plot('voltage_class',ax=ax, figsize=(10, 8), legend=True)

plt.savefig(vis_save_to_root/f"WB6_existing_VREs_and_lines_{country_kwd}.png", dpi=500, bbox_inches='tight')

# Attribute Maps

## Load Capacity and Scores

In [ ]:
if existing_VREs_gdf is not None and not existing_VREs_gdf.empty:
    if existing_VREs_gdf.crs != CRS_m:
        existing_VREs_plot = existing_VREs_gdf.to_crs(CRS_m)
    else:
        existing_VREs_plot = existing_VREs_gdf

### Aggregated Capacity

- Calculate

In [ ]:
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

WB6_cells_aggr = get_sub_nationally_aggregated_capacity(WB6_cells, 
                                                        'Country')
# WB6_cells_sum = WB6_cells.groupby('Country').sum(numeric_only=True)
# Map potential_capacity_solar and potential_capacity_wind to each country and get geometry
WB6_capacity_map = WB6_boundary_dissolved.copy()
WB6_capacity_map["potential_capacity_solar_GW"] = WB6_capacity_map["Country"].map(WB6_cells_aggr["potential_capacity_solar"])/1E3
WB6_capacity_map["potential_capacity_wind_GW"] = WB6_capacity_map["Country"].map(WB6_cells_aggr["potential_capacity_wind"])/1E3
WB6_capacity_map[["Country", "potential_capacity_solar_GW", "potential_capacity_wind_GW", "geometry"]]

- plot

In [ ]:
if WB6_capacity_map.crs != CRS_m:
    WB6_capacity_map_plot = WB6_capacity_map.to_crs(CRS_m)
else:
    WB6_capacity_map_plot = WB6_capacity_map

In [ ]:
import matplotlib.patheffects as pe

# ========= Create subplots =========
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=500)
Country_name_Y_adjustment:float=12E3 #in meters for CRS_m


# ========= Plot solar capacity =========
WB6_capacity_map_plot.plot(
    column="potential_capacity_solar_GW",
    cmap="YlOrRd",
    linewidth=0.8,
    edgecolor="k",
    legend=True,
    ax=ax1,
    legend_kwds={"label": "Solar Potential (GW)", "shrink": 0.7}
)
ax1.set_title("Solar Potential Capacity (GW)", fontsize=15, weight="bold")
ax1.set_axis_off()

# ========= Annotate numbers and country names with white halo =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_solar_GW"]
    # Capacity value
    ax1.annotate(
        f"{val:,.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=11,
        ha="center",
        va="center",
        fontweight="bold",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    # ========= Country name slightly above =========
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )

# ========= Plot wind capacity ==============
WB6_capacity_map_plot.plot(
    column="potential_capacity_wind_GW",
    cmap="BuPu",
    linewidth=0.8,
    edgecolor="k",
    legend=True,
    ax=ax2,
    legend_kwds={"label": "Wind Potential (GW)", "shrink": 0.7}
)
ax2.set_title("Wind Potential Capacity (GW)", fontsize=15, weight="bold")
ax2.set_axis_off()

# ========= Annotate numbers and country names with white halo =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    # Capacity value
    ax2.annotate(
        f"{val:,.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=11,
        ha="center",
        va="center",
        fontweight="bold",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    # ========= Country name slightly above =========
    ax2.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )

# Add existing VREs to the plot

existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.45, 0.88), ncol=2, fontsize=8, frameon=False)

plt.tight_layout()
plt.savefig(vis_save_to_root/f"{country_kwd}_Capacity_by_Country.png", bbox_inches='tight', transparent=False)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# Ensure 'Region' is in the columns for both boundary and cells
# if 'Region' not in boundary.columns:
#     boundary = boundary.reset_index(inplace=True)

# # Assign a number to each region
# boundary['Region_Number'] = range(1, len(boundary) + 1)

# Define custom bins and labels for solar and wind capacity
solar_bins = [20, 30, 50, 70, 80, float('inf')]  # Custom ranges
solar_labels = ['<20','20-30', '30-55','50-70','>80']  # Labels for legend

# Define custom bins and labels for solar and wind capacity
wind_bins = [20, 30, 40, 50, 60, 80, 100, float('inf')]  # Custom ranges
wind_labels = ['<20','20-30', '30-40','40-50','50-60', '60-80', '>100']  # Labels for legend

# Categorize potential_capacity_solar and potential_capacity_wind into bins
WB6_cells['solar_category'] = pd.cut(WB6_cells_plot['lcoe_solar'], bins=solar_bins, labels=solar_labels, include_lowest=True)
WB6_cells['wind_category'] = pd.cut(WB6_cells_plot['lcoe_wind'], bins=wind_bins, labels=wind_labels, include_lowest=True)

# Create figure and axes for side-by-side plotting
fig, (ax1, ax2) = plt.subplots(figsize=(6, 4), ncols=2,dpi=500)
fig.suptitle("Relative Cost Scoring ($/MWh)", fontsize=12, fontweight='bold')
# Set axis off for both subplots
ax1.set_axis_off()
ax2.set_axis_off()

# Shadow effect offset
# shadow_offset = 0.001

# Plot solar map on ax1
# Add shadow effect for solar map
# boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
# boundary.plot(ax=ax1, facecolor='none', edgecolor='gray', linewidth=1.2, alpha=0.3)  # Shadow layer
# boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Plot solar cells
WB6_cells.plot(column='solar_category', ax=ax1, cmap='YlOrRd', legend=False, edgecolor='white',linewidth=0.2,alpha=1,
        #    legend_kwds={'title': "Solar",'title_fontsize':14, 'bbox_to_anchor':(legend_x_ax_offset,legend_y_ax_offset),'fontsize':14,'frameon': False}
           )

# # Plot actual boundary for solar map
# boundary.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.9)

""" 
# Annotate region numbers for solar map
for idx, row in boundary.iterrows():
    centroid = row.geometry.centroid
    ax1.annotate(f"{row['Region_Number']}", 
                 xy=(centroid.x, centroid.y), 
                 ha='center', va='center',
                 fontsize=7, color='black',
                 bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
"""
# Plot wind map on ax2
# Add shadow effect for wind map
# boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
# boundary.plot(ax=ax2, color='None', edgecolor='k', linewidth=0.2, alpha=0.7)  # Shadow layer
# boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Plot wind cells
WB6_cells.plot(column='wind_category', ax=ax2, cmap='BuPu', legend=False, edgecolor='white',linewidth=0.2,alpha=1,
        #    legend_kwds={'title': "Wind", 'title_fontsize':14, 'bbox_to_anchor':(legend_x_ax_offset,legend_y_ax_offset),'fontsize':14,'frameon': False}
           )

# Plot actual boundary for wind map
# boundary.plot(ax=ax2, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.9)
"""
# Annotate region numbers for wind map
for idx, row in boundary.iterrows():
    centroid = row.geometry.centroid
    ax2.annotate(f"{row['Region_Number']}", 
                 xy=(centroid.x, centroid.y), 
                 ha='center', va='center',
                 fontsize=8, color='black',
                 bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
"""
# Adjust layout for cleaner appearance
fig.patch.set_alpha(0)  # Make figure background transparent
# Add annotation to the figure
fig.text(0.5, 0.01, 
         "Note: The Scoring is calculated to reflect Dollar investment required to get an unit of Energy yield (MWh). "
         "\nTo reflect market competitiveness and incentives, the Score ($/MWh) needs financial adjustment factors to be considered on top of it.",
         ha='center', va='center', fontsize=7, color='k', bbox=dict(facecolor='None', edgecolor='k',linewidth=0.2,boxstyle='round,pad=0.5'))

# Add existing VREs to the plot

existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.45, 0.88), ncol=2, fontsize=8, frameon=False)
# Show the side-by-side plot
# vis.add_compass_arrow_custom(ax1,text_offset=0.03)
vis.add_compass_arrow_custom(ax2,x=0.78,text_offset=0.04)
plt.savefig(vis_save_to_root/"solar_wind_score_map.jpg")

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

# Define colormaps
solar_cmap = cm.get_cmap('YlOrRd', len(solar_labels))
wind_cmap = cm.get_cmap('BuPu', len(wind_labels))

# Generate colors for each bin
solar_colors = [mcolors.rgb2hex(solar_cmap(i)) for i in range(len(solar_labels))]
wind_colors = [mcolors.rgb2hex(wind_cmap(i)) for i in range(len(wind_labels))]

# Aggregate potential capacity for each bin
solar_capacity = (
    WB6_cells.groupby('solar_category')['potential_capacity_solar']
    .sum().div(1e3)
    .reindex(solar_labels, fill_value=0)
)
wind_capacity = (
    WB6_cells.groupby('wind_category')['potential_capacity_wind']
    .sum().div(1e3)
    .reindex(wind_labels, fill_value=0)
)

# --- Drop bins with 0 capacity ---
solar_capacity = solar_capacity[solar_capacity > 0]
wind_capacity = wind_capacity[wind_capacity > 0]

solar_colors_filtered = [solar_colors[solar_labels.index(lbl)] for lbl in solar_capacity.index]
wind_colors_filtered = [wind_colors[wind_labels.index(lbl)] for lbl in wind_capacity.index]

### Solar Plot ###
fig1, ax1 = plt.subplots(figsize=(3, 5), dpi=300)
fig1.patch.set_alpha(0)          # transparent figure background
ax1.set_facecolor('none')        # transparent axis background

ax1.bar(solar_capacity.index, solar_capacity.values, color=solar_colors_filtered, edgecolor='none')
ax1.set_ylabel('Solar Potential (GW)', fontsize=12, weight='bold')
ax1.set_xlabel('Relative cost score ($/MWh)', fontsize=12)

ax1.grid(False)  # Remove grid lines

# Clean look
for spine in ax1.spines.values():
    spine.set_visible(False)
ax1.tick_params(bottom=True, left=True, labelsize=14)

plt.savefig('../vis/WesternBalkanRegions/solar_wind_score_bar1.png', transparent=True)

### Wind Plot ###
fig2, ax2 = plt.subplots(figsize=(5, 3), dpi=300)
fig2.patch.set_alpha(0)
ax2.set_facecolor('none')

ax2.bar(wind_capacity.index, wind_capacity.values, color=wind_colors_filtered, edgecolor='none')
ax2.set_ylabel('Wind Potential (GW)', fontsize=14, weight='bold')
ax2.set_xlabel('Relative cost score ($/MWh)', fontsize=14)

ax2.grid(False)  # Remove grid lines

for spine in ax2.spines.values():
    spine.set_visible(False)
ax2.tick_params(bottom=True, left=True, labelsize=14)

plt.savefig('../vis/WesternBalkanRegions/solar_wind_score_bar2.png', transparent=True)


### Aggregated Capacity with LCOE thresholds

In [ ]:
WB6_cells_plot_solar=WB6_cells_plot[WB6_cells_plot['lcoe_solar'] <= 60]
WB6_cells_plot_wind=WB6_cells_plot[WB6_cells_plot['lcoe_wind'] <= 80]
solar_capacity_haircut=0.8
wind_capacity_haircut=0.6

In [ ]:
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

WB6_cells_solar_aggr = get_sub_nationally_aggregated_capacity(WB6_cells_plot_solar, 
                                                        'Country')
WB6_cells_wind_aggr = get_sub_nationally_aggregated_capacity(WB6_cells_plot_wind, 
                                                        'Country')
# WB6_cells_sum = WB6_cells.groupby('Country').sum(numeric_only=True)
# Map potential_capacity_solar and potential_capacity_wind to each country and get geometry
WB6_capacity_with_threshold_map = WB6_boundary_dissolved_reproj.copy()
WB6_capacity_with_threshold_map["potential_capacity_solar_GW"] = WB6_capacity_with_threshold_map["Country"].map(WB6_cells_solar_aggr["potential_capacity_solar"])/1E3 * solar_capacity_haircut
WB6_capacity_with_threshold_map["potential_capacity_wind_GW"] = WB6_capacity_with_threshold_map["Country"].map(WB6_cells_wind_aggr["potential_capacity_wind"])/1E3*wind_capacity_haircut
WB6_capacity_with_threshold_map[["Country", "potential_capacity_solar_GW", "potential_capacity_wind_GW", "geometry"]]

#### Assumption 2 no haircuts

In [ ]:
WB6_cells_plot_solar=WB6_cells_plot[WB6_cells_plot['lcoe_solar'] <= 65]
WB6_cells_plot_wind=WB6_cells_plot[WB6_cells_plot['lcoe_wind'] <= 80]
solar_capacity_haircut=0.8
wind_capacity_haircut=0.6
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

WB6_cells_solar_aggr = get_sub_nationally_aggregated_capacity(WB6_cells_plot_solar, 
                                                        'Country')
WB6_cells_wind_aggr = get_sub_nationally_aggregated_capacity(WB6_cells_plot_wind, 
                                                        'Country')
# WB6_cells_sum = WB6_cells.groupby('Country').sum(numeric_only=True)
# Map potential_capacity_solar and potential_capacity_wind to each country and get geometry
WB6_capacity_with_threshold_map = WB6_boundary_dissolved_reproj.copy()
WB6_capacity_with_threshold_map["potential_capacity_solar_GW"] = WB6_capacity_with_threshold_map["Country"].map(WB6_cells_solar_aggr["potential_capacity_solar"])/1E3 * solar_capacity_haircut
WB6_capacity_with_threshold_map["potential_capacity_wind_GW"] = WB6_capacity_with_threshold_map["Country"].map(WB6_cells_wind_aggr["potential_capacity_wind"])/1E3*wind_capacity_haircut
WB6_capacity_with_threshold_map[["Country", "potential_capacity_solar_GW", "potential_capacity_wind_GW", "geometry"]]

- plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# =============================
# 1️⃣ Create Figure and Subplots
# =============================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), dpi=1000)
# fig.suptitle(f"Resources and Capacity by Country – {country_name}", fontsize=18, fontweight='bold')

# =============================
# 2️⃣ Plot resource maps (from first block)
# =============================
vis.get_data_in_map_plot(
    WB6_cells_plot_solar, 
    resource_type='solar',
    datafield='score',
    compass_size=12,
    ax=ax1, 
    score_threshold=250,
    show=False
)

vis.get_data_in_map_plot(
    WB6_cells_plot_wind, 
    resource_type='wind',
    datafield='score',
    ax=ax2, 
    score_threshold=250,
    show=False
)

# =============================
# 3️⃣ Overlay “Cells without suitable land”
# =============================
WB6_cells_plot.plot(ax=ax1, color='gray', alpha=0.7, zorder=1)
WB6_cells_plot.plot(ax=ax2, color='gray', alpha=0.6, zorder=1)

# Create legend patch
no_land_patch = mpatches.Patch(
    facecolor='gray',
    edgecolor='lightgray',
    alpha=0.7,
    label='economically unfeasible or no developable land'
)

# =============================
# 4️⃣ Add existing solar/wind projects
# =============================
existing_VREs_gdf_solar = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'solar']
ax1, solar_legends = vis.get_existing_committed_VRE_plot(
    ax=ax1,
    existing_VREs_gdf=existing_VREs_gdf_solar,
    existing_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=14,
    marker_highlight_width=3
)

existing_VREs_gdf_wind = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'wind']
ax2, wind_legends = vis.get_existing_committed_VRE_plot(
    ax=ax2,
    existing_VREs_gdf=existing_VREs_gdf_wind,
    existing_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=14,
    marker_highlight_width=3
)

# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================
Country_name_Y_adjustment = 12E3  # meters in CRS_m


# Add text annotations for capacity and country name
for idx, row in WB6_capacity_with_threshold_map.iterrows():
    centroid = row.geometry.centroid

    # Solar capacity on left
    ax1.annotate(
        f"{row['potential_capacity_solar_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

    # Wind capacity on right
    ax2.annotate(
        f"{row['potential_capacity_wind_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax2.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

# =============================
# 6️⃣ Add legends and note
# =============================
all_legends = solar_legends + wind_legends + [no_land_patch]
fig.legend(
    handles=all_legends,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.88),
    ncol=1,
    fontsize=8,
    frameon=False
)
# Plot only boundaries (no fill)
WB6_capacity_with_threshold_map.boundary.plot(ax=ax1, color='black', linewidth=0.9, zorder=6)
WB6_capacity_with_threshold_map.boundary.plot(ax=ax2, color='black', linewidth=0.6, zorder=6)

plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_and_Capacity_Combined_with_LCOE_Thresholds.png", bbox_inches='tight', transparent=False)


### Capacity Factor

* Individual Maps

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), dpi=1000)

vis.get_data_in_map_plot(WB6_cells_plot, 
                resource_type='solar',
                datafield='CF',
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(WB6_cells_plot, 
                resource_type='wind',
                datafield='CF',
                ax=ax2, 
                show=False)
# ========= Country name slightly above =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    ax2.annotate(
            row["Country"],
            (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
            color="black",
            fontsize=10,
            ha="center",
            va="bottom",
            fontweight="normal",
            path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
        )
# Add existing VREs to the plot

existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    target_crs=CRS_m,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    marker_scale_existing=15,
                                                    marker_highlight_width=2)

existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                     marker_scale_existing=14,
                                                    marker_highlight_width=2)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.5, 0.8), ncol=1, fontsize=8, frameon=False)

vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)
plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CF.png", bbox_inches='tight', transparent=False)

### Capacity

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), dpi=1000)
# for double coulmn figure at 500 Dpi (72points per inch) minimum wide required is 3750 px, hence figsize=(7.5, 5) is recommended

# fig.suptitle(f"Resources for {country_name}", fontsize=18, fontweight='bold')

vis.get_data_in_map_plot(WB6_cells_plot, 
                resource_type='solar',
                datafield='capacity',
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(WB6_cells_plot, 
                resource_type='wind',
                datafield='capacity',
                ax=ax2, 
                show=False)
# ========= Country name slightly above =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    ax2.annotate(
            row["Country"],
            (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
            color="black",
            fontsize=10,
            ha="center",
            va="bottom",
            fontweight="normal",
            path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
        )
# Add existing VREs to the plot
existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.5, 0.88), ncol=1, fontsize=8, frameon=False)

# vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)
plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CAPACITY.png", bbox_inches='tight', transparent=False)

### Score

In [ ]:
WB6_cells_clean_solar = WB6_cells_plot[(WB6_cells_plot['potential_capacity_solar'] >= 1) &(WB6_cells_plot['solar_CF_mean'] > 0)]
WB6_cells_clean_wind = WB6_cells_plot[(WB6_cells_plot['potential_capacity_wind'] >= 3) &(WB6_cells_plot['wind_CF_mean'] > 0)]

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), dpi=1000)
# for double coulmn figure at 500 Dpi (72points per inch) minimum wide required is 3750 px, hence figsize=(7.5, 5) is recommended

fig.suptitle(f"Resources for {country_name}", fontsize=18, fontweight='bold')

vis.get_data_in_map_plot(WB6_cells_clean_solar, 
                resource_type='solar',
                datafield='score',
                compass_size=12,
                ax=ax1, 
                score_threshold=250,
                show=False)

vis.get_data_in_map_plot(WB6_cells_clean_wind, 
                resource_type='wind',
                datafield='score',
                ax=ax2, 
                score_threshold=250,
                show=False,)
# ========= Country name slightly above =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    ax2.annotate(
            row["Country"],
            (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
            color="black",
            fontsize=10,
            ha="center",
            va="bottom",
            fontweight="normal",
            path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
        )
fig.text(
    0.5, -0.05,
    "Note: The Scoring is calculated to reflect Dollar investment required to get a unit of Energy yield (MWh).To reflect market competitiveness and incentives, the Score (CAD/MWh) needs financial adjustment factors to be considered on top of it. Score higher than 200 $/MWh are assumed to be not feasible and not shown in this map.",
    ha='center', va='top', fontsize=7, color='gray',wrap=True,
)# Add existing VREs to the plot
existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)

# Add all cells in light grey to indicate area without suitable land
WB6_cells_plot.plot(ax=ax1, color='gray', alpha=1,zorder=1) 
WB6_cells_plot.plot(ax=ax2, color='gray', alpha=1,zorder=1)

# Create legend patch for cells without suitable land
no_land_patch = mpatches.Patch(
    facecolor='gray',
    edgecolor='lightgray',
    alpha=0.5,
    label='Cells without suitable land'
)

# Combine with existing handles
all_legends = solar_legends + wind_legends + [no_land_patch]

fig.legend(handles=all_legends, loc='upper center', bbox_to_anchor=(0.5, 0.88), ncol=1, fontsize=8, frameon=False)

plt.tight_layout()

plt.savefig(f"../vis/{country_kwd}/Resources_combined_SCORE.png", bbox_inches='tight', transparent=False)

- Score with Capacity

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# =============================
# 1️⃣ Create Figure and Subplots
# =============================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), dpi=1000)
# fig.suptitle(f"Resources and Capacity by Country – {country_name}", fontsize=18, fontweight='bold')

# =============================
# 2️⃣ Plot resource maps (from first block)
# =============================
vis.get_data_in_map_plot(
    WB6_cells_clean_solar, 
    resource_type='solar',
    datafield='score',
    compass_size=12,
    ax=ax1, 
    score_threshold=250,
    show=False
)

vis.get_data_in_map_plot(
    WB6_cells_clean_wind, 
    resource_type='wind',
    datafield='score',
    ax=ax2, 
    score_threshold=250,
    show=False
)

# =============================
# 3️⃣ Overlay “Cells without suitable land”
# =============================
WB6_cells_plot.plot(ax=ax1, color='gray', alpha=1, zorder=1)
WB6_cells_plot.plot(ax=ax2, color='gray', alpha=1, zorder=1)

# Create legend patch
no_land_patch = mpatches.Patch(
    facecolor='gray',
    edgecolor='lightgray',
    alpha=0.5,
    label='Cells without suitable land'
)

# =============================
# 4️⃣ Add existing solar/wind projects
# =============================
existing_VREs_gdf_solar = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'solar']
ax1, solar_legends = vis.get_existing_committed_VRE_plot(
    ax=ax1,
    existing_VREs_gdf=existing_VREs_gdf_solar,
    existing_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=14,
    marker_highlight_width=3
)

existing_VREs_gdf_wind = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'wind']
ax2, wind_legends = vis.get_existing_committed_VRE_plot(
    ax=ax2,
    existing_VREs_gdf=existing_VREs_gdf_wind,
    existing_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=14,
    marker_highlight_width=3
)

# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================
Country_name_Y_adjustment = 12E3  # meters in CRS_m

# Plot only boundaries (no fill)
WB6_capacity_map_plot.boundary.plot(ax=ax1, color='black', linewidth=0.6, zorder=3)
WB6_capacity_map_plot.boundary.plot(ax=ax2, color='black', linewidth=0.6, zorder=3)

# Add text annotations for capacity and country name
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid

    # Solar capacity on left
    ax1.annotate(
        f"{row['potential_capacity_solar_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

    # Wind capacity on right
    ax2.annotate(
        f"{row['potential_capacity_wind_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax2.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

# =============================
# 6️⃣ Add legends and note
# =============================
all_legends = solar_legends + wind_legends + [no_land_patch]
fig.legend(
    handles=all_legends,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.88),
    ncol=1,
    fontsize=8,
    frameon=False
)

fig.text(
    0.5, -0.05,
    "Note: The Scoring reflects relative investment per MWh yield. Values above 250 $/MWh are considered non-feasible. "
    "Country-level potentials (in GW) are annotated from aggregated site capacities.",
    ha='center',
    va='top',
    fontsize=9,
    color='gray',
    wrap=True,
)

plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_and_Capacity_Combined.png", bbox_inches='tight', transparent=False)
